# ML, Deep Learning & NLP Applications - Capstone Project

This notebook follows the assignment brief: data loading, exploration, preprocessing, three ML models, a neural network, and a short comparison report.

In [ ]:

import os
import re
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

plt.style.use('default')


## Step 0 - Load the dataset

In [ ]:

data_dir = 'data'
fake = pd.read_csv(os.path.join(data_dir, 'Fake.csv'))
true = pd.read_csv(os.path.join(data_dir, 'True.csv'))

fake['label'] = 0
true['label'] = 1

df = pd.concat([fake, true], ignore_index=True)
df['label_name'] = df['label'].map({0: 'fake', 1: 'true'})
df['combined_text'] = (df['title'].fillna('') + ' ' + df['text'].fillna('')).str.lower()
df['combined_text'] = df['combined_text'].str.replace(r'[^a-z\s]', ' ', regex=True).str.replace(r'\s+', ' ', regex=True).str.strip()

df.head()


## Step 1 - Explore the data

In [ ]:

print('Shape:', df.shape)
print('
Missing values:
', df[['title', 'text', 'subject', 'date']].isna().sum())
print('
Class balance:
', df['label_name'].value_counts())
print('
Subject counts:
', df['subject'].value_counts().head(8))


In [ ]:

fig, ax = plt.subplots(figsize=(6, 4))
df['label_name'].value_counts().plot(kind='bar', ax=ax)
ax.set_title('Class Distribution')
ax.set_xlabel('Class')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()


## Step 2 - Prepare the text features

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    df['combined_text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

vectorizer = TfidfVectorizer(max_features=500, stop_words='english', min_df=5, max_df=0.9)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print('Train shape:', X_train_tfidf.shape)
print('Test shape:', X_test_tfidf.shape)


## Step 3 - Train three machine-learning models

In [ ]:

models = {
    'Logistic Regression': LogisticRegression(max_iter=500, solver='liblinear'),
    'Multinomial Naive Bayes': MultinomialNB(),
    'Linear SVM (SGDClassifier)': SGDClassifier(loss='hinge', max_iter=1000, tol=1e-3, random_state=42),
}

results = []
conf_mats = {}

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    pred = model.predict(X_test_tfidf)
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred),
        'F1-score': f1_score(y_test, pred),
    })
    conf_mats[name] = confusion_matrix(y_test, pred)

results_df = pd.DataFrame(results).sort_values('F1-score', ascending=False).reset_index(drop=True)
results_df


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, cm) in zip(axes, conf_mats.items()):
    im = ax.imshow(cm, cmap='Blues')
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['0', '1'])
    ax.set_yticklabels(['0', '1'])

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f'{cm[i, j]}', ha='center', va='center', color='black', fontsize=12)

fig.suptitle('Confusion Matrices - Step 1 Models', fontsize=16, y=1.02)
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8, pad=0.02)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()


## Step 4 - Neural network

In [ ]:

# Balanced sample for a fast dense neural network in this environment
sample = pd.concat([
    df[df['label'] == 0].sample(1000, random_state=42),
    df[df['label'] == 1].sample(1000, random_state=42),
], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

Xn_train, Xn_test, yn_train, yn_test = train_test_split(
    sample['combined_text'], sample['label'], test_size=0.2, random_state=42, stratify=sample['label']
)

nn_vectorizer = TfidfVectorizer(max_features=300, stop_words='english', min_df=2, max_df=0.9)
Xn_train = nn_vectorizer.fit_transform(Xn_train).astype(np.float32).toarray()
Xn_test = nn_vectorizer.transform(Xn_test).astype(np.float32).toarray()
yn_train = np.array(yn_train.values).reshape(-1, 1).astype(np.float32)
yn_test = np.array(yn_test.values).reshape(-1, 1).astype(np.float32)


In [ ]:

rng = np.random.default_rng(42)
input_dim = Xn_train.shape[1]
h1, h2 = 128, 64
W1 = rng.normal(0, np.sqrt(2 / input_dim), (input_dim, h1)).astype(np.float32)
b1 = np.zeros((1, h1), np.float32)
W2 = rng.normal(0, np.sqrt(2 / h1), (h1, h2)).astype(np.float32)
b2 = np.zeros((1, h2), np.float32)
W3 = rng.normal(0, np.sqrt(2 / h2), (h2, 1)).astype(np.float32)
b3 = np.zeros((1, 1), np.float32)

mW1 = np.zeros_like(W1); vW1 = np.zeros_like(W1)
mb1 = np.zeros_like(b1); vb1 = np.zeros_like(b1)
mW2 = np.zeros_like(W2); vW2 = np.zeros_like(W2)
mb2 = np.zeros_like(b2); vb2 = np.zeros_like(b2)
mW3 = np.zeros_like(W3); vW3 = np.zeros_like(W3)
mb3 = np.zeros_like(b3); vb3 = np.zeros_like(b3)

def relu(x):
    return np.maximum(0, x)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def bce(y, p):
    eps = 1e-7
    p = np.clip(p, eps, 1 - eps)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p)).mean()

def adam_update(param, grad, m, v, t, lr=0.003, beta1=0.9, beta2=0.999, eps=1e-8):
    m = beta1 * m + (1 - beta1) * grad
    v = beta2 * v + (1 - beta2) * (grad * grad)
    m_hat = m / (1 - beta1 ** t)
    v_hat = v / (1 - beta2 ** t)
    param = param - lr * m_hat / (np.sqrt(v_hat) + eps)
    return param, m, v

val_split = 0.1
val_n = int(len(Xn_train) * val_split)
Xn_tr, Xn_val = Xn_train[:-val_n], Xn_train[-val_n:]
yn_tr, yn_val = yn_train[:-val_n], yn_train[-val_n:]

history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}
t = 0
for epoch in range(20):
    perm = rng.permutation(len(Xn_tr))
    Xs = Xn_tr[perm]
    ys = yn_tr[perm]
    for i in range(0, len(Xs), 128):
        xb = Xs[i:i+128]
        yb = ys[i:i+128]
        z1 = xb @ W1 + b1
        a1 = relu(z1)
        mask1 = (rng.random(a1.shape) > 0.2).astype(np.float32) / 0.8
        a1d = a1 * mask1
        z2 = a1d @ W2 + b2
        a2 = relu(z2)
        mask2 = (rng.random(a2.shape) > 0.1).astype(np.float32) / 0.9
        a2d = a2 * mask2
        z3 = a2d @ W3 + b3
        p = sigmoid(z3)

        m = len(xb)
        dz3 = (p - yb) / m
        dW3 = a2d.T @ dz3
        db3 = dz3.sum(axis=0, keepdims=True)
        da2d = dz3 @ W3.T
        da2 = da2d * mask2
        dz2 = da2 * (z2 > 0)
        dW2 = a1d.T @ dz2
        db2 = dz2.sum(axis=0, keepdims=True)
        da1d = dz2 @ W2.T
        da1 = da1d * mask1
        dz1 = da1 * (z1 > 0)
        dW1 = xb.T @ dz1
        db1 = dz1.sum(axis=0, keepdims=True)

        t += 1
        W1, mW1, vW1 = adam_update(W1, dW1, mW1, vW1, t)
        b1, mb1, vb1 = adam_update(b1, db1, mb1, vb1, t)
        W2, mW2, vW2 = adam_update(W2, dW2, mW2, vW2, t)
        b2, mb2, vb2 = adam_update(b2, db2, mb2, vb2, t)
        W3, mW3, vW3 = adam_update(W3, dW3, mW3, vW3, t)
        b3, mb3, vb3 = adam_update(b3, db3, mb3, vb3, t)

    train_pred = sigmoid(relu(relu(Xn_tr @ W1 + b1) @ W2 + b2) @ W3 + b3)
    val_pred = sigmoid(relu(relu(Xn_val @ W1 + b1) @ W2 + b2) @ W3 + b3)
    history['loss'].append(float(bce(yn_tr, train_pred)))
    history['val_loss'].append(float(bce(yn_val, val_pred)))
    history['accuracy'].append(float(((train_pred >= 0.5) == yn_tr).mean()))
    history['val_accuracy'].append(float(((val_pred >= 0.5) == yn_val).mean()))

nn_pred = sigmoid(relu(relu(Xn_test @ W1 + b1) @ W2 + b2) @ W3 + b3)
nn_labels = (nn_pred >= 0.5).astype(int)

nn_results = pd.DataFrame([{
    'Model': 'Neural Network (balanced sample)',
    'Accuracy': accuracy_score(yn_test, nn_labels),
    'Precision': precision_score(yn_test, nn_labels),
    'Recall': recall_score(yn_test, nn_labels),
    'F1-score': f1_score(yn_test, nn_labels),
}])

final_results = pd.concat([results_df, nn_results], ignore_index=True).sort_values('F1-score', ascending=False).reset_index(drop=True)
final_results


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, len(history['loss']) + 1)
axes[0].plot(epochs, history['loss'], label='Training Loss')
axes[0].plot(epochs, history['val_loss'], label='Validation Loss')
axes[0].set_title('Loss Curves')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[1].plot(epochs, history['accuracy'], label='Training Accuracy')
axes[1].plot(epochs, history['val_accuracy'], label='Validation Accuracy')
axes[1].set_title('Accuracy Curves')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
plt.tight_layout()
plt.show()


## Step 5 - Save outputs

In [ ]:

results_df.to_csv('model_results.csv', index=False)
final_results.to_csv('final_results.csv', index=False)
print('Saved CSV files and plots to the project folder.')
